# Falcon-H1 steering pilot: post-convergence rerouting in a pretrained parallel hybrid

**Question.** Falcon-H1-0.5B-Base already solves in-context key–value recall, and the measurement
notebook showed that attention carries the retrieved value. If we now make attention *unreliable*
during a short fine-tune, does the model move recall into its SSM branch while keeping accuracy,
and does that reverse when the pressure is removed?

**Design.** One seed; a pilot, not the final experiment.
- **Starting point:** the pretrained model. It is converged on the task before anything is
  changed.
- **What trains:** only the SSM (Mamba) branch in every layer, or optionally the SSM plus the
  MLPs. Attention weights stay frozen, so any change in how much the model relies on attention
  comes from the rest of the network learning to work without it.
- **Intervention:** attention availability, the paper's no-penalty control. On each step, a
  fraction `P_ABSENT` of prompts get their attention output set to zero in every layer. This
  intervention cannot be Goodharted the way a norm price can in a fused model.
- **Data:** recall prompts at the training loads (2 and 4 pairs), fresh random pairs every step,
  plus a small language-model replay batch from WikiText-2 *train* on every step, without any
  intervention, so fine-tuning cannot erase general text ability. Perplexity is measured on the
  validation split. Evaluation uses fixed prompts at 2, 4 and 8 pairs, where 8 is never trained.

**Arms.**

| arm | start | steps | attention during training |
|---|---|---|---|
| `never` | pretrained | `STEPS` | always available |
| `late` | pretrained | `STEPS` | absent on `P_ABSENT` of prompts |
| `withdrawn` | end of `late`, with its optimiser state | `WITHDRAW_STEPS` | always available |

`never` controls for fine-tuning itself: it trains the same parameters on the same data with no
pressure.

**Measurements at each evaluation point** (per load, fixed prompts).
- **Clean accuracy.**
- **No-attention accuracy:** can the model answer with attention deleted? This is the most direct
  sign that the SSM has taken over the answer.
- **Donor rate:** with attention resampled from another prompt, how often does the model output
  that prompt's answer? A fall means it has stopped following attention.
- **Route dependence** D_attn and D_ssm, normalised by in-context chance (1/L).
- **WikiText-2 perplexity** at the start and end of each arm, to check that general language
  modelling is preserved.

**Success criteria.** Written to `pilot_criteria.json` before any training, and never
overwritten.
- **Rerouted:** at the end, averaged over training loads, `late` keeps clean accuracy within
  `ACC_TOL` of `never`, keeps WikiText perplexity within `PPL_TOL`× of the pretrained model, *and*
  either no-attention accuracy is at least `NOATTN_GAIN` higher than `never`, or the donor rate is
  at least `DONOR_DROP` lower.
- **Reversed:** `withdrawn` gives back at least half of that change.

**Runtime.** On a T4 the model trains in float32 on transformers' slow PyTorch SSM path. The batch
benchmark cell prints seconds per step and an estimate of the total. Expect several hours for the
full pilot; an L4 or A100 (bfloat16 autocast) is much faster.

**Checkpoints.** Mid-run checkpoints hold the trainable weights and optimiser state, about 2 GB
each on Drive, every `CKPT_MINUTES`. The final state of `late` is kept because `withdrawn` starts
from it; everything else is deleted when its arm finishes.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import hashlib, json, math, os, pathlib, random, subprocess, sys, time
from collections import defaultdict
from contextlib import contextmanager, nullcontext

# ============================== CHOOSE HERE ==============================
PRESET     = "full"     # "smoke": Falcon-H1-Tiny-90M, a few steps | "full": Falcon-H1-0.5B-Base pilot
MODEL_ID   = None
TRAINABLE  = "ssm"       # "ssm": SSM branch only | "ssm+mlp": SSM branch and MLPs
USE_DRIVE  = True
DRIVE_DIR  = os.environ.get("RESULTS_DIR", "results/falcon_h1_steering")
# =========================================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("FALCON_STEER_WORKDIR", str(pathlib.Path.cwd() / "falcon_h1_steering"))

from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return tuple(int("".join(ch for ch in part if ch.isdigit()) or 0) for part in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


if _ver("transformers") < (4, 53):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    raise SystemExit("transformers was upgraded. Runtime -> Restart session, then Run all again.")
if _ver("datasets") == (0, 0):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


def P(full, smoke):
    return full if PRESET == "full" else smoke


MODEL = MODEL_ID or P("tiiuae/Falcon-H1-0.5B-Base", "tiiuae/Falcon-H1-Tiny-90M-Instruct")
TRAIN_LOADS    = [2, 4]
EVAL_LOADS     = [2, 4, 8]                  # 8 is never trained on
STEPS          = P(200, 20)                 # never and late
WITHDRAW_STEPS = P(100, 10)
EVAL_AT        = P([0, 25, 50, 100, 150, 200], [0, 10, 20])
EVAL_AT_WD     = P([0, 25, 50, 100], [0, 5, 10])
P_ABSENT       = 0.5
LR             = 2e-5                       # 1e-4 destroyed language modelling within 20 steps in smoke
LM_W           = 1.0                        # weight of the language-model replay loss
LM_REPLAY_BATCH, LM_REPLAY_LEN = 4, 128     # WikiText-2 *train* windows per step (perplexity uses validation)
WARMUP         = P(20, 2)
GRAD_CLIP      = 1.0
TRAIN_BATCH    = 8                   # or an int
BATCH_CANDIDATES = [8, 16, 32, 64]
EVAL_N         = P(64, 16)                  # fixed prompts per load
EVAL_BATCH     = 16
N_PPL_WINDOWS  = P(16, 2)
PPL_LEN        = 256
PPL_BATCH      = 8
CKPT_MINUTES   = P(20.0, 2.0)
SEED, EVAL_SEED, FORMAT_VERSION = 0, 12345, 2
D_FLOOR = 0.10

# success criteria (pre-registered; means over TRAIN_LOADS at the end of each arm)
ACC_TOL, NOATTN_GAIN, DONOR_DROP, REVERSAL_FRACTION = 0.05, 0.30, 0.20, 0.5
PPL_TOL = 1.5                               # late clean perplexity must stay within 1.5x of the pretrained model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
USE_BF16 = IS_CUDA and torch.cuda.get_device_capability()[0] >= 8   # float32 weights + bf16 autocast


def amp():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


TAG = f"{MODEL.split('/')[-1]}_{PRESET}_{TRAINABLE.replace('+', '_')}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, CKPTDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in ("results", "checkpoints", "figures", "tables"))
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"colab {IN_COLAB} | {GPU_NAME} | {'bf16 autocast' if USE_BF16 else 'float32'} | torch {torch.__version__} "
      f"| transformers {transformers.__version__}")
print(f"preset {PRESET} | {MODEL} | trainable {TRAINABLE} | steps {STEPS}+{WITHDRAW_STEPS} | p_absent {P_ABSENT}")
print(f"work dir {WORK}")


In [ ]:
def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


def ckpt_path(name):
    return CKPTDIR / f"{name}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    tmp, prev = p.with_suffix(".tmp"), p.with_suffix(".prev")
    torch.save(obj, tmp)
    if p.exists():
        os.replace(p, prev)
    os.replace(tmp, p)


def ckpt_load(name):
    p = ckpt_path(name)
    for cand in (p, p.with_suffix(".prev")):
        if cand.exists():
            try:
                return torch.load(cand, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"  checkpoint {cand.name} unreadable ({type(e).__name__}); trying the previous one")
    return None


def ckpt_delete(name):
    p = ckpt_path(name)
    for cand in (p, p.with_suffix(".prev"), p.with_suffix(".tmp")):
        if cand.exists():
            cand.unlink()


REC = Records(RESDIR / "records.jsonl")
BASE_SPEC = {"model": MODEL, "seed": SEED, "format": FORMAT_VERSION, "trainable": TRAINABLE,
             "eval_n": EVAL_N, "eval_loads": EVAL_LOADS, "bf16": USE_BF16}
print(f"{len(REC.recs)} finished record(s) on disk")


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False


def find_layers(m):
    n = m.config.num_hidden_layers
    lists = [(name, mod) for name, mod in m.named_modules() if isinstance(mod, torch.nn.ModuleList) and len(mod) == n]
    assert lists, f"no ModuleList of length {n}"
    return lists[0]


def find_branch(layer, preferred, patterns):
    for attr in preferred:
        mod = getattr(layer, attr, None)
        if isinstance(mod, torch.nn.Module):
            return attr, mod
    hits = [(n, c) for n, c in layer.named_children() if any(p in type(c).__name__.lower() for p in patterns)]
    assert len(hits) == 1, f"cannot identify {patterns}: {[(n, type(c).__name__) for n, c in layer.named_children()]}"
    return hits[0]


LAYER_PATH, LAYERS = find_layers(model)
ATTN = [find_branch(l, ("self_attn",), ("attention",)) for l in LAYERS]
SSM = [find_branch(l, ("mamba",), ("mamba", "mixer", "ssm")) for l in LAYERS]
N_LAYERS, HIDDEN = len(LAYERS), model.config.hidden_size

# ---- trainable parameters ----------------------------------------------------------------------
for p in model.parameters():
    p.requires_grad_(False)
_train_modules = [mod for _, mod in SSM]
if TRAINABLE == "ssm+mlp":
    _train_modules += [l.feed_forward for l in LAYERS if hasattr(l, "feed_forward")]
for mod in _train_modules:
    for p in mod.parameters():
        p.requires_grad_(True)
TRAIN_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
TRAIN_PARAMS = [p for p in model.parameters() if p.requires_grad]
N_TRAIN = sum(p.numel() for p in TRAIN_PARAMS)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters, {N_TRAIN / 1e6:.1f}M trainable "
      f"({TRAINABLE}) | {N_LAYERS} layers | attention .{ATTN[0][0]} ({type(ATTN[0][1]).__name__}) | "
      f"SSM .{SSM[0][0]} ({type(SSM[0][1]).__name__})")

try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    GRAD_CKPT = True
except Exception as e:
    GRAD_CKPT = False
    print(f"gradient checkpointing unavailable ({type(e).__name__}); training uses more memory")


def trainable_state():
    return {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(state):
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state:
                p.copy_(state[n].to(p.device))


INIT_STATE = trainable_state()          # pretrained values of the trainable parameters (CPU)

# ---- route interventions (forward hooks) ---------------------------------------------------------
# {"kind": "record", "store": list} | {"kind": "zero"} | {"kind": "cache", "store": per-layer, "perm": idx}
# | {"kind": "mask", "keep": bool[B]}   (availability: zero the branch output for rows where keep is False)
ACTIVE = {}


def _hook(route, idx):
    def fn(module, inputs, output):
        spec = ACTIVE.get(route)
        if spec is None:
            return None
        t = output[0] if isinstance(output, tuple) else output
        if not torch.is_tensor(t):
            raise TypeError(f"unexpected {route} output {type(output)}")
        kind = spec["kind"]
        if kind == "record":
            spec["store"].append(t.detach().clone())
            return None
        if kind == "zero":
            new = torch.zeros_like(t)
        elif kind == "cache":
            new = spec["store"][idx][spec["perm"]]
        elif kind == "mask":
            new = t * spec["keep"].to(t.dtype)[:, None, None]
        else:
            raise ValueError(kind)
        return (new,) + tuple(output[1:]) if isinstance(output, tuple) else new
    return fn


for _h in globals().get("HOOK_HANDLES", []):
    _h.remove()
HOOK_HANDLES = ([mod.register_forward_hook(_hook("attn", i)) for i, (_, mod) in enumerate(ATTN)]
                + [mod.register_forward_hook(_hook("ssm", i)) for i, (_, mod) in enumerate(SSM)])


@contextmanager
def intervene(**routes):
    ACTIVE.clear()
    ACTIVE.update({k: v for k, v in routes.items() if v is not None})
    try:
        yield
    finally:
        ACTIVE.clear()


def _forward_last(x):
    for kw in ({"logits_to_keep": 1}, {"num_logits_to_keep": 1}, {}):
        try:
            return model(input_ids=x, use_cache=False, **kw).logits[:, -1, :].float()
        except TypeError:
            continue
    raise RuntimeError("model forward failed")


@torch.no_grad()
def last_logits(x):
    with amp():
        return _forward_last(x)


In [ ]:
KEY_WORDS = """apple river table window garden doctor pencil island forest bottle mirror candle bridge castle
rocket tiger lemon piano jacket helmet market pocket village teacher engine planet carpet button ticket basket
hammer ladder wallet bucket camera coffee cookie dragon farmer finger flower guitar hunter kitten letter monkey
needle orange parrot pillow puzzle rabbit saddle sister spider statue sugar thunder tomato turtle wagon wizard
anchor arrow badge banner barrel beach beard bell belt bench blanket boat bone book boot bread brick brush cabin
cable cake canal cannon card cart chain chair cheese cherry chest chicken church cloud coat coin comet cotton
crown cup curtain desk diamond dinner dog door drum eagle egg elbow fence field flag flute fork fox frog glass
glove goat grape hat honey horse house iron jar jewel kettle key kite knife lake lamp leaf lion lock map mask
milk moon mouse nail nest net oven owl paint palace paper pearl pepper pig pipe plate pond pot queen rain ring
road robot roof rope rose salt scarf ship shirt shoe shovel sink skirt snake soap sock spoon star stone storm
sword tent tower train tree truck tunnel violin wall whale wheel wolf yard""".split()
VALUE_WORDS = """red blue green yellow purple brown black white pink gray golden silent happy angry brave calm
clever cold dark eager early empty fair fancy fast fierce gentle giant grand great heavy hollow humble icy jolly
kind large late lazy little lonely loud lucky mild modern narrow neat noble odd old plain polite proud quick
quiet rapid rare rich rough round royal rusty sacred safe sharp shiny short shy simple sleepy slow small smooth
soft solid sour spicy steep sticky stormy strange strict strong sunny sweet swift tall tame thick thin tidy tiny
tough ugly vast warm weak wet wide wild wise young bitter bold bright broad busy cheap crisp cruel cute damp deep
dry dusty faint false fresh frozen full funny grumpy hairy harsh hot huge juicy keen lean lively loyal mad merry
messy misty muddy nasty naughty nervous nice pale pure rigid ripe rude salty scary shallow silly skinny spotty
stale stiff sturdy tender tense tricky vivid weird wooden zany""".split()


def single_token_ids(words, exclude=()):
    ids = []
    for w in words:
        e = tok.encode(" " + w, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids and e[0] not in exclude:
            ids.append(e[0])
    return ids


KEY_IDS = single_token_ids(KEY_WORDS)
VAL_IDS = single_token_ids(VALUE_WORDS, exclude=set(KEY_IDS))
assert max(EVAL_LOADS) <= min(len(KEY_IDS), len(VAL_IDS))
POOL = torch.tensor(VAL_IDS, device=DEVICE)
CHANCE = 1.0 / len(VAL_IDS)


def piece(s):
    return tok.encode(s, add_special_tokens=False)


BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
INTRO, RECALL = piece("Memorize the following pairs.\n"), piece("Now recall one of them.\n")
DASH, COLON, NEWLINE = piece("-"), piece(":"), piece("\n")


def make_prompts(load, n, rng):
    X, Y, CTX = [], [], []
    for _ in range(n):
        ks = rng.choice(len(KEY_IDS), load, replace=False)
        vs = rng.choice(len(VAL_IDS), load, replace=False)
        q = int(rng.integers(load))
        ids = BOS + INTRO
        for k, v in zip(ks, vs):
            ids += DASH + [KEY_IDS[k]] + COLON + [VAL_IDS[v]] + NEWLINE
        ids += RECALL + DASH + [KEY_IDS[ks[q]]] + COLON
        X.append(ids); Y.append(VAL_IDS[vs[q]]); CTX.append([VAL_IDS[v] for v in vs])
    return torch.tensor(X), torch.tensor(Y), torch.tensor(CTX)


EVAL_SETS = {L: make_prompts(L, EVAL_N, np.random.default_rng([EVAL_SEED, L, FORMAT_VERSION])) for L in EVAL_LOADS}
print(f"pools: {len(KEY_IDS)} keys, {len(VAL_IDS)} values | fixed evaluation prompts: {EVAL_N} per load {EVAL_LOADS}")
print("example:\n" + tok.decode(EVAL_SETS[2][0][0]) + " -> " + tok.decode([int(EVAL_SETS[2][1][0])]))


In [ ]:
def slices(n, B):
    B = max(2, B)
    sl = [(s, min(s + B, n)) for s in range(0, n, B)]
    if len(sl) > 1 and sl[-1][1] - sl[-1][0] == 1:
        sl[-2] = (sl[-2][0], n); sl.pop()
    return sl


def with_oom_retry(fn, B, label):
    while True:
        try:
            return fn(B)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear()
            if IS_CUDA:
                torch.cuda.empty_cache()
            if B <= 2:
                raise
            B //= 2
            print(f"    out of memory in {label}; retrying with batch {B}", flush=True)


def score(lg, y, ctx, y_donor):
    pool_pred = POOL[lg[:, POOL].argmax(-1)]
    ctx_pred = ctx.gather(1, torch.gather(lg, 1, ctx).argmax(-1, keepdim=True)).squeeze(1)
    return {"acc": float((pool_pred == y).sum()), "acc_ctx": float((ctx_pred == y).sum()),
            "bag": float((pool_pred[:, None] == ctx).any(1).sum()), "donor": float((pool_pred == y_donor).sum())}


def _dep(clean, intervened, chance):
    if clean - chance < D_FLOOR:
        return float("nan")
    return float(np.clip((clean - intervened) / (clean - chance), 0.0, 1.0))


@torch.no_grad()
def eval_recall(B=EVAL_BATCH):
    was_training = model.training
    model.eval()
    out = {}
    for L, (X, Y, CTX) in EVAL_SETS.items():
        tot = defaultdict(float)
        for i, j in slices(len(X), B):
            x, y, ctx = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), CTX[i:j].to(DEVICE)
            perm = torch.roll(torch.arange(x.shape[0], device=DEVICE), 1)
            y_d = y[perm]
            rec = {"attn": [], "ssm": []}
            with intervene(attn={"kind": "record", "store": rec["attn"]}, ssm={"kind": "record", "store": rec["ssm"]}):
                runs = {"clean": last_logits(x)}
            with intervene(attn={"kind": "cache", "store": rec["attn"], "perm": perm}):
                runs["attn_res"] = last_logits(x)
            with intervene(ssm={"kind": "cache", "store": rec["ssm"], "perm": perm}):
                runs["ssm_res"] = last_logits(x)
            del rec
            with intervene(attn={"kind": "zero"}):
                runs["no_attn"] = last_logits(x)
            with intervene(ssm={"kind": "zero"}):
                runs["no_ssm"] = last_logits(x)
            for c, lg in runs.items():
                for k, v in score(lg, y, ctx, y_d).items():
                    tot[f"{c}_{k}"] += v
        n = len(X)
        m = {k: v / n for k, v in tot.items()}
        for k, v in m.items():
            out[f"{k}_L{L}"] = v
        out[f"D_attn_ctx_L{L}"] = _dep(m["clean_acc_ctx"], m["attn_res_acc_ctx"], 1.0 / L)
        out[f"D_ssm_ctx_L{L}"] = _dep(m["clean_acc_ctx"], m["ssm_res_acc_ctx"], 1.0 / L)
        if IS_CUDA:
            torch.cuda.empty_cache()
    if was_training:
        model.train()
    return out


def _wikitext(split):
    from datasets import load_dataset
    try:
        return load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split=split)
    except Exception:
        return load_dataset("wikitext", "wikitext-2-raw-v1", split=split)


_LM_TRAIN = None


def replay_loss(rng):
    # next-token loss on generic text, with no intervention, to keep language modelling intact
    global _LM_TRAIN
    if _LM_TRAIN is None:
        ids = tok.encode("\n\n".join(t for t in _wikitext("train")["text"] if t.strip()), add_special_tokens=False)
        n = len(ids) // LM_REPLAY_LEN
        _LM_TRAIN = torch.tensor(ids[: n * LM_REPLAY_LEN]).view(n, LM_REPLAY_LEN)
    x = _LM_TRAIN[rng.integers(0, len(_LM_TRAIN), LM_REPLAY_BATCH)].to(DEVICE)
    with amp():
        lg = model(input_ids=x, use_cache=False).logits[:, :-1]
    return F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1))


_PPL_X = None


@torch.no_grad()
def measure_ppl(B=PPL_BATCH):
    global _PPL_X
    if _PPL_X is None:
        ds = _wikitext("validation")
        ids = tok.encode("\n\n".join(t for t in ds["text"] if t.strip()), add_special_tokens=False)
        _PPL_X = torch.tensor([ids[i:i + PPL_LEN] for i in range(0, len(ids) - PPL_LEN, PPL_LEN)][:N_PPL_WINDOWS])
    was_training = model.training
    model.eval()
    out = {}
    for cond, routes in [("clean", {}), ("no_attn", {"attn": {"kind": "zero"}})]:
        nll, n = 0.0, 0
        for i in range(0, len(_PPL_X), B):
            x = _PPL_X[i:i + B].to(DEVICE)
            with intervene(**routes), amp():
                lg = model(input_ids=x, use_cache=False).logits[:, :-1].float()
            nll += float(F.cross_entropy(lg.reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1), reduction="sum"))
            n += x[:, 1:].numel()
            del lg
        out[f"ppl_{cond}"] = math.exp(nll / n)
    if was_training:
        model.train()
    return out


def summary_line(ev, loads=EVAL_LOADS):
    return "  ".join(f"L{L}: acc {ev[f'clean_acc_L{L}']:.2f} noAttn {ev[f'no_attn_acc_L{L}']:.2f} "
                     f"donor {ev[f'attn_res_donor_L{L}']:.2f} Ds {ev[f'D_ssm_ctx_L{L}']:.2f}" for L in loads)


In [ ]:
# Self-test: hooks, availability mask, gradients reach only the trainable branch, state round trip.
model.eval()
_X, _Y, _C = EVAL_SETS[EVAL_LOADS[0]]
_x, _y = _X[:4].to(DEVICE), _Y[:4].to(DEVICE)
_rec = {"attn": [], "ssm": []}
with intervene(attn={"kind": "record", "store": _rec["attn"]}, ssm={"kind": "record", "store": _rec["ssm"]}):
    _clean = last_logits(_x)
assert len(_rec["attn"]) == N_LAYERS == len(_rec["ssm"]), "hooks did not fire once per layer"
_tol = (1e-4 if not USE_BF16 else 5e-3) * max(1.0, float(_clean.abs().max()))
_id = torch.arange(4, device=DEVICE)
with intervene(attn={"kind": "cache", "store": _rec["attn"], "perm": _id}, ssm={"kind": "cache", "store": _rec["ssm"], "perm": _id}):
    assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "identity resample changed the output"
with intervene(attn={"kind": "mask", "keep": torch.ones(4, dtype=torch.bool, device=DEVICE)}):
    assert float((last_logits(_x) - _clean).abs().max()) <= _tol, "all-True mask changed the output"
with intervene(attn={"kind": "zero"}):
    _za = last_logits(_x)
with intervene(attn={"kind": "mask", "keep": torch.zeros(4, dtype=torch.bool, device=DEVICE)}):
    assert float((last_logits(_x) - _za).abs().max()) <= _tol, "all-False mask differs from deletion"
assert float((_za - _clean).abs().max()) > 10 * _tol, "deleting attention had no effect"
del _rec

model.train()
_keep = torch.tensor([True, False, True, False], device=DEVICE)
with intervene(attn={"kind": "mask", "keep": _keep}):
    with amp():
        _lg = _forward_last(_x)
    _loss = F.cross_entropy(_lg, _y)
    _loss.backward()
_with_grad = [n for n, p in model.named_parameters() if p.grad is not None and float(p.grad.abs().sum()) > 0]
_frozen_grad = [n for n, p in model.named_parameters() if not p.requires_grad and p.grad is not None]
assert not _frozen_grad, f"frozen parameters received gradients: {_frozen_grad[:3]}"
assert _with_grad, "no trainable parameter received a gradient"
assert set(_with_grad) <= set(TRAIN_NAMES)
_frac = len(_with_grad) / len(TRAIN_NAMES)
for p in TRAIN_PARAMS:
    p.grad = None
_s = trainable_state()
load_trainable(_s)
assert all(torch.equal(p.detach().cpu(), _s[n]) for n, p in model.named_parameters() if n in _s)
model.eval()
if IS_CUDA:
    torch.cuda.empty_cache()
print(f"self-test passed: hooks x{N_LAYERS}, identity resample, availability mask, gradients only in the "
      f"trainable branch ({len(_with_grad)}/{len(TRAIN_NAMES)} tensors non-zero), state round trip "
      f"| gradient checkpointing {GRAD_CKPT}")


In [ ]:
# Pick the training batch: seconds per step and prompts per second at each candidate size
# (forward + backward, no optimiser step, so the weights do not change).
def bench(B, reps=2):
    rng = np.random.default_rng(999)
    model.train()
    times = []
    for _ in range(reps):
        x, y, _ = make_prompts(max(TRAIN_LOADS), B, rng)
        x, y = x.to(DEVICE), y.to(DEVICE)
        keep = torch.rand(B, device=DEVICE) >= P_ABSENT
        sync(); t0 = time.time()
        with intervene(attn={"kind": "mask", "keep": keep}):
            with amp():
                lg = _forward_last(x)
            F.cross_entropy(lg, y).backward()
        if LM_W > 0:
            (LM_W * replay_loss(rng)).backward()
        sync(); times.append(time.time() - t0)
        for p in TRAIN_PARAMS:
            p.grad = None
    model.eval()
    return times[-1]


bench_spec = {**BASE_SPEC, "block": "bench", "candidates": BATCH_CANDIDATES, "gpu": GPU_NAME,
              "replay": [LM_W, LM_REPLAY_BATCH, LM_REPLAY_LEN]}
BENCH = REC.get(bench_spec)
if BENCH is None and TRAIN_BATCH == "auto":
    rows = []
    for B in BATCH_CANDIDATES:
        try:
            s = bench(B)
            rows.append({"batch": B, "sec_per_step": s, "prompts_per_sec": B / s})
            print(f"  batch {B:>3}: {s:.2f} s/step, {B / s:.1f} prompts/s", flush=True)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear(); torch.cuda.empty_cache()
            print(f"  batch {B:>3}: out of memory")
            break
    BENCH = REC.put(bench_spec, {"rows": rows})
if TRAIN_BATCH == "auto":
    best = max(BENCH["rows"], key=lambda r: r["prompts_per_sec"])
    # prefer the smallest batch within 15% of the best throughput (more optimiser steps per prompt seen)
    TB = min(r["batch"] for r in BENCH["rows"] if r["prompts_per_sec"] >= 0.85 * best["prompts_per_sec"])
    SEC_PER_STEP = next(r["sec_per_step"] for r in BENCH["rows"] if r["batch"] == TB)
else:
    TB, SEC_PER_STEP = int(TRAIN_BATCH), float("nan")
_train_steps = 2 * STEPS + WITHDRAW_STEPS
print(f"training batch {TB} | about {SEC_PER_STEP:.1f} s/step -> {_train_steps} steps is about "
      f"{SEC_PER_STEP * _train_steps / 3600:.1f} h of training, plus evaluations")


In [ ]:
CRIT_PATH = RESDIR / "pilot_criteria.json"
if not CRIT_PATH.exists():
    CRIT_PATH.write_text(json.dumps({
        "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "model": MODEL, "trainable": TRAINABLE,
        "train_loads": TRAIN_LOADS, "eval_loads": EVAL_LOADS, "steps": STEPS, "withdraw_steps": WITHDRAW_STEPS,
        "p_absent": P_ABSENT, "lr": LR, "lm_replay": [LM_W, LM_REPLAY_BATCH, LM_REPLAY_LEN],
        "rerouted": f"late clean acc >= never - {ACC_TOL}, late clean perplexity <= {PPL_TOL} x pretrained, "
                    f"and (late no-attn acc >= never + {NOATTN_GAIN} "
                    f"or late donor rate <= never - {DONOR_DROP}); means over train loads at the end",
        "reversed": f"withdrawn gives back >= {REVERSAL_FRACTION} of the late-vs-never change in whichever "
                    f"metric met its threshold"}, indent=1))
print(f"criteria recorded {json.loads(CRIT_PATH.read_text())['created']}")

base_spec = {**BASE_SPEC, "block": "base"}
BASE = REC.get(base_spec)
if BASE is None:
    t0 = time.time()
    ev = with_oom_retry(lambda B: eval_recall(B), EVAL_BATCH, "base evaluation")
    ppl = with_oom_retry(lambda B: measure_ppl(B), PPL_BATCH, "base perplexity")
    BASE = REC.put(base_spec, {"eval": ev, "ppl": ppl, "seconds": time.time() - t0})
print(f"pretrained model ({BASE['seconds']:.0f} s): {summary_line(BASE['eval'])}")
print(f"  perplexity clean {BASE['ppl']['ppl_clean']:.2f}, without attention {BASE['ppl']['ppl_no_attn']:.2f}")


In [ ]:
def arm_spec(name, steps, p_absent, init_key):
    return {**BASE_SPEC, "block": "arm", "arm": name, "steps": steps, "p_absent": p_absent, "lr": LR,
            "batch": TB, "train_loads": TRAIN_LOADS, "init": init_key, "warmup": WARMUP,
            "replay": [LM_W, LM_REPLAY_BATCH, LM_REPLAY_LEN]}


def train_arm(name, steps, p_absent, eval_at, init_state, init_opt, init_key, step_offset=0, keep_final=False):
    spec = arm_spec(name, steps, p_absent, init_key)
    key = keyof(spec)
    done = REC.get(spec)
    if done is not None and (not keep_final or ckpt_path(f"final_{key}").exists()):
        print(f"[{name}] finished earlier")
        return done, key
    fin = ckpt_load(f"final_{key}")
    if fin is None:
        load_trainable(init_state)
        opt = torch.optim.AdamW(TRAIN_PARAMS, lr=LR, betas=(0.9, 0.95), weight_decay=0.0)
        if init_opt is not None:
            opt.load_state_dict(init_opt)
        start, trace, run_time, run_steps = 0, [], 0.0, 0
        ck = ckpt_load(f"run_{key}")
        if ck is not None:
            load_trainable(ck["trainable"]); opt.load_state_dict(ck["opt"])
            start, trace, run_time, run_steps = ck["step"], ck["trace"], ck["run_time"], ck["run_steps"]
            print(f"[{name}] resumed at step {start}/{steps}")
        if start == 0:
            ev = with_oom_retry(lambda B: eval_recall(B), EVAL_BATCH, f"{name} eval")
            trace.append({"step": step_offset, "train_loss": float("nan"), "loss_absent": float("nan"),
                          "train_lm": float("nan"), **ev})
            print(f"[{name}] step {step_offset}: {summary_line(ev)}", flush=True)
        rng = np.random.default_rng([SEED, int(key[:8], 16), start])
        gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2 ** 62)))
        model.train()
        last_ck, losses, absent, lm_losses = time.time(), [], [], []
        for step in range(start + 1, steps + 1):
            sync(); t_step = time.time()
            if init_opt is None:
                for g in opt.param_groups:
                    g["lr"] = LR * min(1.0, step / WARMUP)
            L = TRAIN_LOADS[step % len(TRAIN_LOADS)]
            x, y, _ = make_prompts(L, TB, rng)
            x, y = x.to(DEVICE), y.to(DEVICE)
            keep = (torch.rand(TB, generator=gen, device=DEVICE) >= p_absent) if p_absent > 0 else None
            routes = {"attn": {"kind": "mask", "keep": keep}} if keep is not None else {}
            opt.zero_grad(set_to_none=True)
            with intervene(**routes):
                with amp():
                    lg = _forward_last(x)
                row_loss = F.cross_entropy(lg, y, reduction="none")
                loss = row_loss.mean()
                loss.backward()
            if LM_W > 0:
                lm_loss = replay_loss(rng)
                (LM_W * lm_loss).backward()
                lm_losses.append(float(lm_loss.detach()))
            torch.nn.utils.clip_grad_norm_(TRAIN_PARAMS, GRAD_CLIP)
            opt.step()
            sync(); run_time += time.time() - t_step; run_steps += 1
            losses.append(float(loss.detach()))
            if keep is not None and bool((~keep).any()):
                absent.append(float(row_loss.detach()[~keep].mean()))
            if step == start + 3:
                print(f"[{name}] {run_time / run_steps:.1f} s/step, about "
                      f"{run_time / run_steps * (steps - step) / 60:.0f} min of training left", flush=True)
            if step in eval_at:
                ev = with_oom_retry(lambda B: eval_recall(B), EVAL_BATCH, f"{name} eval")
                trace.append({"step": step_offset + step, "train_loss": float(np.mean(losses)),
                              "loss_absent": float(np.mean(absent)) if absent else float("nan"),
                              "train_lm": float(np.mean(lm_losses)) if lm_losses else float("nan"), **ev})
                print(f"[{name}] step {step_offset + step} (recall loss {np.mean(losses):.3f}"
                      + (f", attention-absent rows {np.mean(absent):.3f}" if absent else "")
                      + (f", text loss {np.mean(lm_losses):.3f}" if lm_losses else "")
                      + f"): {summary_line(ev)}", flush=True)
                losses, absent, lm_losses = [], [], []
                model.train()
            if step < steps and time.time() - last_ck > 60 * CKPT_MINUTES:
                ckpt_save(f"run_{key}", {"trainable": trainable_state(), "opt": opt.state_dict(), "step": step,
                                         "trace": trace, "run_time": run_time, "run_steps": run_steps})
                last_ck = time.time()
        fin = {"trainable": trainable_state(), "opt": opt.state_dict() if keep_final else None, "trace": trace,
               "sec_per_step": run_time / max(run_steps, 1)}
        ckpt_save(f"final_{key}", fin)
        ckpt_delete(f"run_{key}")
        del opt
    else:
        load_trainable(fin["trainable"])
        print(f"[{name}] final state found on disk")
    ppl = with_oom_retry(lambda B: measure_ppl(B), PPL_BATCH, f"{name} perplexity")
    rec = REC.put(spec, {"trace": fin["trace"], "ppl": ppl, "sec_per_step": fin["sec_per_step"]})
    if not keep_final:
        ckpt_delete(f"final_{key}")
    print(f"[{name}] perplexity clean {ppl['ppl_clean']:.2f}, without attention {ppl['ppl_no_attn']:.2f}")
    if IS_CUDA:
        torch.cuda.empty_cache()
    return rec, key


NEVER, _ = train_arm("never", STEPS, 0.0, EVAL_AT, INIT_STATE, None, "pretrained")
LATE, LATE_KEY = train_arm("late", STEPS, P_ABSENT, EVAL_AT, INIT_STATE, None, "pretrained", keep_final=True)
_late_final = ckpt_load(f"final_{LATE_KEY}")
assert _late_final is not None and _late_final.get("opt") is not None, "late final state missing"
WITHDRAWN, _ = train_arm("withdrawn", WITHDRAW_STEPS, 0.0, EVAL_AT_WD, _late_final["trainable"],
                         _late_final["opt"], LATE_KEY, step_offset=STEPS)
del _late_final


In [ ]:
ARMS = [("never", NEVER), ("late", LATE), ("withdrawn", WITHDRAWN)]
HELD = [L for L in EVAL_LOADS if L not in TRAIN_LOADS]
METRICS = [("clean_acc", "recall accuracy"), ("no_attn_acc", "accuracy with attention deleted"),
           ("attn_res_donor", "donor answer under attention resampling"), ("D_attn_ctx", "attention dependence"),
           ("D_ssm_ctx", "SSM dependence"), ("ssm_res_acc", "accuracy with SSM resampled")]


def _mean(row, metric, loads):
    vals = [row.get(f"{metric}_L{L}", np.nan) for L in loads]
    vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
    return float(np.mean(vals)) if vals else float("nan")


rows = []
for arm, rec in [("pretrained", {"trace": [{"step": 0, **BASE["eval"]}]})] + ARMS:
    for t in rec["trace"]:
        r = {"arm": arm, "step": t["step"], "train_loss": t.get("train_loss", np.nan),
             "loss_absent": t.get("loss_absent", np.nan), "train_lm": t.get("train_lm", np.nan)}
        for m, _ in METRICS:
            r[f"{m}_train"] = _mean(t, m, TRAIN_LOADS)
            r[f"{m}_held"] = _mean(t, m, HELD)
        rows.append(r)
TR = pd.DataFrame(rows)
TR.to_csv(TABDIR / "steering_trajectories.csv", index=False)
END = {arm: TR[TR.arm == arm].sort_values("step").iloc[-1] for arm in ["pretrained", "never", "late", "withdrawn"]}
PPLS = {"pretrained": BASE["ppl"], "never": NEVER["ppl"], "late": LATE["ppl"], "withdrawn": WITHDRAWN["ppl"]}

# ---- pre-registered criteria --------------------------------------------------------------------
nv, lt, wd = END["never"], END["late"], END["withdrawn"]
acc_ok = lt["clean_acc_train"] >= nv["clean_acc_train"] - ACC_TOL
ppl_ratio = LATE["ppl"]["ppl_clean"] / BASE["ppl"]["ppl_clean"]
ppl_ok = ppl_ratio <= PPL_TOL
gain = lt["no_attn_acc_train"] - nv["no_attn_acc_train"]
drop = nv["attn_res_donor_train"] - lt["attn_res_donor_train"]
noattn_ok, donor_ok = gain >= NOATTN_GAIN, drop >= DONOR_DROP
rev_noattn = (lt["no_attn_acc_train"] - wd["no_attn_acc_train"]) / gain if gain > 0 else float("nan")
rev_donor = (wd["attn_res_donor_train"] - lt["attn_res_donor_train"]) / drop if drop > 0 else float("nan")
REROUTED = bool(acc_ok and ppl_ok and (noattn_ok or donor_ok))
PARTIAL = bool(acc_ok and ppl_ok and (gain >= NOATTN_GAIN / 2 or drop >= DONOR_DROP / 2))
REVERSED = bool((noattn_ok and rev_noattn >= REVERSAL_FRACTION) or (donor_ok and rev_donor >= REVERSAL_FRACTION))
VERDICT = ("REROUTED" if REROUTED else "PARTIAL SHIFT" if PARTIAL else
           "BEHAVIOUR NOT PRESERVED" if not (acc_ok and ppl_ok) else "NO SHIFT")

# ---- figure -----------------------------------------------------------------------------------
mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.labelsize": 8, "axes.titlesize": 8.5, "legend.fontsize": 7, "xtick.labelsize": 7,
                     "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False})
COL = {"never": "#1b6ca8", "late": "#e07a1f", "withdrawn": "#d1495b"}
fig, axes = plt.subplots(2, 3, figsize=(7.4, 4.2), sharex=True)
for ax, (m, title) in zip(axes.ravel(), METRICS):
    for arm in ["never", "late", "withdrawn"]:
        d = TR[TR.arm == arm].sort_values("step")
        ax.plot(d.step, d[f"{m}_train"], "-o", ms=2.5, color=COL[arm], label=f"{arm}, loads {TRAIN_LOADS}")
        if HELD:
            ax.plot(d.step, d[f"{m}_held"], ":", marker="s", ms=2, color=COL[arm], label=f"{arm}, load {HELD}")
    ax.scatter([0], [END["pretrained"][f"{m}_train"]], color="k", s=12, zorder=5, label="pretrained")
    ax.axvline(STEPS, color="0.6", lw=0.6, ls=":")
    ax.set_ylim(-0.03, 1.03); ax.set_title(title)
for ax in axes[1]:
    ax.set_xlabel("fine-tuning step")
h, lab = axes[0, 0].get_legend_handles_labels()
fig.tight_layout()
fig.legend(h, lab, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
for ext in ("pdf", "png"):
    fig.savefig(FIGDIR / f"falcon_steering_pilot.{ext}")
plt.show()

# ---- table ------------------------------------------------------------------------------------
def f3(v):
    return "--" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}"


TAB = pd.DataFrame([{"state": s, **{f"{m}_{w}": END[s][f"{m}_{w}"] for m, _ in METRICS for w in ("train", "held")},
                     "ppl_clean": PPLS[s]["ppl_clean"], "ppl_no_attn": PPLS[s]["ppl_no_attn"]} for s in END])
TAB.to_csv(TABDIR / "steering_endpoints.csv", index=False)
tex = [r"% generated by falcon_h1_steering_pilot.ipynb -- requires booktabs", r"\begin{table}[t]", r"\centering", r"\small",
       rf"\caption{{\textbf{{Post-convergence steering pilot in Falcon-H1-0.5B-Base}} (one seed). Only the "
       rf"{'SSM branch' if TRAINABLE == 'ssm' else 'SSM branch and MLPs'} is fine-tuned ({N_TRAIN / 1e6:.0f}M "
       rf"parameters); attention is frozen. \emph{{late}}: attention output removed on "
       rf"{int(100 * P_ABSENT)}\% of prompts for {STEPS} steps; \emph{{never}}: same fine-tuning without "
       rf"removal; \emph{{withdrawn}}: {WITHDRAW_STEPS} further steps without removal. Recall metrics are means "
       rf"over {TRAIN_LOADS} pairs (held-out {HELD} in parentheses).}}",
       r"\label{tab:falcon-steer}", r"\begin{tabular}{lccccccc}", r"\toprule",
       r"state & acc. & no-attn acc. & donor & $D_a$ & $D_s$ & PPL & PPL no-attn \\", r"\midrule"]
for s in ["pretrained", "never", "late", "withdrawn"]:
    e = END[s]
    cells = [s] + [f"{f3(e[f'{m}_train'])} ({f3(e[f'{m}_held'])})" for m in ("clean_acc", "no_attn_acc", "attn_res_donor",
                                                                           "D_attn_ctx", "D_ssm_ctx")]
    cells += [f"{PPLS[s]['ppl_clean']:.2f}", f"{PPLS[s]['ppl_no_attn']:.2f}"]
    tex.append(" & ".join(cells) + r" \\")
tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(TABDIR / "falcon_steering_table.tex").write_text("\n".join(tex))

# ---- summary ----------------------------------------------------------------------------------
crit = json.loads(CRIT_PATH.read_text())
show = ["state"] + [c for c in TAB.columns if c != "state"]
lines = [f"=== FALCON-H1 STEERING PILOT | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"model {MODEL} | {GPU_NAME} | {'bf16 autocast' if USE_BF16 else 'float32'} | trainable {TRAINABLE} "
         f"({N_TRAIN / 1e6:.1f}M) | batch {TB} | lr {LR} | p_absent {P_ABSENT} | steps {STEPS}+{WITHDRAW_STEPS}",
         f"sec/step: never {NEVER['sec_per_step']:.1f}, late {LATE['sec_per_step']:.1f}, "
         f"withdrawn {WITHDRAWN['sec_per_step']:.1f} | criteria recorded {crit['created']}",
         "", "end states (train-load mean; held-out load in *_held):", TAB[show].round(3).T.to_string(header=False),
         "", "trajectories (train-load means):",
         TR[["arm", "step", "train_loss", "loss_absent", "train_lm", "clean_acc_train", "no_attn_acc_train",
             "attn_res_donor_train", "D_attn_ctx_train", "D_ssm_ctx_train", "no_attn_acc_held"]].round(3).to_string(index=False),
         "", f"late vs never: clean-accuracy change {lt['clean_acc_train'] - nv['clean_acc_train']:+.3f} "
             f"(tolerance -{ACC_TOL}) | no-attention accuracy gain {gain:+.3f} (threshold {NOATTN_GAIN}) | "
             f"donor-rate drop {drop:+.3f} (threshold {DONOR_DROP})",
         f"late clean perplexity {LATE['ppl']['ppl_clean']:.2f} = {ppl_ratio:.2f}x pretrained (limit {PPL_TOL}x); "
         f"never {NEVER['ppl']['ppl_clean'] / BASE['ppl']['ppl_clean']:.2f}x",
         f"withdrawal gives back: no-attention {f3(rev_noattn)}, donor {f3(rev_donor)} (threshold {REVERSAL_FRACTION})",
         f"VERDICT: {VERDICT} | REVERSED: {REVERSED}"]
SUMMARY = "\n".join(lines)
(TABDIR / "STEERING_SUMMARY.txt").write_text(SUMMARY)
print(SUMMARY)


## Reading the pilot

**The two signals that matter.**
- **No-attention accuracy in `late` rising well above `never`:** the SSM branch has learned to
  answer on its own.
- **Donor rate in `late` falling below `never`:** the model has stopped simply copying what
  attention writes.

**Why dependence alone can mislead here.** A resampled attention output injects a *confident wrong
answer*. It can keep D_attn high even after the SSM has learned the task, which is the override
effect discussed for the additive model. So D_attn is reported, but it is not a criterion.

**What each verdict supports.**
- **REROUTED and REVERSED:** a post-convergence, reversible shift of which branch carries recall in
  a pretrained 0.5B parallel hybrid, with attention frozen, under a no-penalty intervention. One
  seed makes it a pilot; say so, and add seeds in the revision.
- **PARTIAL SHIFT:** dependence moves without meeting the thresholds. Report it as a pilot trend.
- **NO SHIFT:** within this budget the SSM does not take over recall, consistent with the
  measurement result that the routes are composed. That is informative, but it does not support a
  steering claim.

**Also check.**
- **`never` itself:** fine-tuning alone should not move the no-attention accuracy.
- **Perplexity:** it should stay close to the pretrained value.
- **The held-out load of 8 pairs:** whether any shift generalises beyond the trained loads.
